# **General Steps involved**

* EDA-to understand the data
* Train-Test split
* Transformers for data preprocessing: Always pass names of the columns and not a data frame
* Build model pipelines
* Compare models: Cross Validation
* Hyperparameter tuning
* Select best model
* Pickle the model

In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

In [38]:
data=sns.load_dataset('iris')
data.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [39]:
data['sepal_length'].describe()

,sepal_length
count,150.000000
mean,5.843333
std,0.828066
min,4.300000
25%,5.100000
50%,5.800000
75%,6.400000
max,7.900000


In [40]:
# Creating a categorical column
data['speal_bins']=pd.cut(data['sepal_length'],bins=[4,6,8],labels=['Small','Large'])
data.head()

,sepal_length,sepal_width,petal_length,petal_width,species,speal_bins
0,5.1,3.5,1.4,0.2,setosa,Small
1,4.9,3.0,1.4,0.2,setosa,Small
2,4.7,3.2,1.3,0.2,setosa,Small
3,4.6,3.1,1.5,0.2,setosa,Small
4,5.0,3.6,1.4,0.2,setosa,Small


**Train Test split**

In [41]:
from sklearn.model_selection import train_test_split
X=data.drop('species',axis=1)
y=data['species']
X_temp,X_test,y_temp,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train,X_val,y_train,y_val=train_test_split(X_temp,y_temp,test_size=0.1)

In [42]:
X_train.shape

(108, 5)

In [43]:
X_train.head()

,sepal_length,sepal_width,petal_length,petal_width,speal_bins
93,5.0,2.3,3.3,1.0,Small
136,6.3,3.4,5.6,2.4,Large
134,6.1,2.6,5.6,1.4,Large
103,6.3,2.9,5.6,1.8,Large
14,5.8,4.0,1.2,0.2,Small


**Column Transformer**
* Create pipeline for numerical data
* Create pipeline for categorical data
* Combine them in a column transformer

In [58]:
from sklearn.preprocessing import StandardScaler,OrdinalEncoder
from sklearn.impute import SimpleImputer

num_data = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
cat_data = ['speal_bins']

num_pipeline=Pipeline([
    ('scaler',StandardScaler())
])

cat_pipeline=Pipeline([
    ('encoder',OrdinalEncoder())
])

preprocessor=ColumnTransformer([
    ('num',num_pipeline,num_data),
    ('cat',cat_pipeline,cat_data)
],remainder='passthrough')

**Pipelines with models**

In [69]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier,AdaBoostClassifier

pipelines={
'pipe_decision_tree':Pipeline([
    ('preprocessor',preprocessor),
    ('model',DecisionTreeClassifier())
]),

'pipe_random_forest':Pipeline([
    ('preprocessor',preprocessor),
    ('model',RandomForestClassifier())
]),

'pipe_gradient_boost':Pipeline([
    ('preprocessor',preprocessor),
    ('model',GradientBoostingClassifier())
]),

'pipe_ada_boost':Pipeline([
    ('preprocessor',preprocessor),
    ('model',AdaBoostClassifier())
])
}

**Train the pipelines**

In [70]:
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    score = pipe.score(X_test, y_test)
    print(f"{name}: {score:.4f}")


pipe_decision_tree: 1.0000
pipe_random_forest: 1.0000
pipe_gradient_boost: 1.0000
pipe_ada_boost: 0.9667


**Cross validation**

In [71]:
from sklearn.model_selection import cross_val_score

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    val_score = pipe.score(X_val, y_val)
    print(f"{name} Validation score: {val_score:.4f}")

pipe_decision_tree Validation score: 1.0000
pipe_random_forest Validation score: 0.9167
pipe_gradient_boost Validation score: 1.0000
pipe_ada_boost Validation score: 0.9167


**Hyperparameter tuning**

In [72]:
param_grids = {
    'pipe_decision_tree': {
        'model__max_depth': [3, 5, 7, None],
        'model__min_samples_split': [2, 5, 10]
    },
    'pipe_random_forest': {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5]
    },
    'pipe_gradient_boost': {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.01, 0.1, 0.2],
        'model__max_depth': [3, 5]
    },
    'pipe_ada_boost': {
        'model__n_estimators': [50, 100, 200],
        'model__learning_rate': [0.01, 0.1, 1.0]
    }
}

In [73]:
from sklearn.model_selection import GridSearchCV

best_models = {}

for name, pipe in pipelines.items():
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    print(f"{name} Best Params: {grid.best_params_}")
    print(f"{name} CV Score: {grid.best_score_:.4f}")

    # Evaluate on validation set
    val_score = grid.best_estimator_.score(X_val, y_val)
    print(f"{name} Validation Score: {val_score:.4f}\n")

    best_models[name] = (grid.best_estimator_, val_score)

pipe_decision_tree Best Params: {'model__max_depth': 3, 'model__min_samples_split': 2}
pipe_decision_tree CV Score: 0.9532
pipe_decision_tree Validation Score: 0.9167

pipe_random_forest Best Params: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 300}
pipe_random_forest CV Score: 0.9437
pipe_random_forest Validation Score: 0.9167

pipe_gradient_boost Best Params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}
pipe_gradient_boost CV Score: 0.9442
pipe_gradient_boost Validation Score: 1.0000

pipe_ada_boost Best Params: {'model__learning_rate': 0.1, 'model__n_estimators': 200}
pipe_ada_boost CV Score: 0.9437
pipe_ada_boost Validation Score: 0.9167



In [80]:
pipelines['pipe_gradient_boost'].fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['sepal_length',
                                                   'sepal_width',
                                                   'petal_length',
                                                   'petal_width']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OrdinalEncoder())]),
                                                  ['speal_bins'])])),
                ('model', GradientBoostingClassifier())])

In [94]:
print(grid.best_estimator_)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['sepal_length',
                                                   'sepal_width',
                                                   'petal_length',
                                                   'petal_width']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OrdinalEncoder())]),
                                                  ['speal_bins'])])),
                ('model',
                 AdaBoostClassifier(learning_rate=0.1, n_estimators=200))])


**Updating a pipeline**

In [104]:
# Replace the model step with tuned hyperparameters
pipelines['pipe_gradient_boost'].set_params(
    model=GradientBoostingClassifier(learning_rate=0.1, max_depth=3, n_estimators=200)
)

# Print the pipeline again
print(pipelines['pipe_gradient_boost'])

#Since learning_rate=0.1 and max_depth=3 are default and hence they are not displayed

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['sepal_length',
                                                   'sepal_width',
                                                   'petal_length',
                                                   'petal_width']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OrdinalEncoder())]),
                                                  ['speal_bins'])])),
                ('model', GradientBoostingClassifier(n_estimators=200))])


**Pickle**

In [99]:
print(grid.best_estimator_)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['sepal_length',
                                                   'sepal_width',
                                                   'petal_length',
                                                   'petal_width']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OrdinalEncoder())]),
                                                  ['speal_bins'])])),
                ('model',
                 AdaBoostClassifier(learning_rate=0.1, n_estimators=200))])


In [90]:
import pickle

with open('iris_classifier.pkl','wb') as f:
  pickle.dump(grid.best_estimator_,f)

In [91]:
with open('iris_classifier.pkl','rb') as f:
  model=pickle.load(f)

In [93]:
from sklearn.metrics import accuracy_score
y_pred=model.predict(X_test)
print(accuracy_score(y_pred,y_test))

1.0
